# Building bobabean's review-reply agent with LangGraph

The original bobabean pipeline (`../generate_replies.py`) makes **one** Claude call per review: read the review, get sentiment + a drafted reply back, done.

This notebook rebuilds that as a small **multi-agent** pipeline instead — four small "agents" (just Python functions, each making one Claude call) wired into a graph with LangGraph:

```
triage --> analyst --> drafter <--> critic --> ship
   |                                    |
   v                                    v
escalate                            escalate
(unsafe for AI to reply)      (still rejected after 2 tries)
```

We build and test each agent on its own first, *then* wire them into a graph at the end — the same "get one small piece working before combining it" approach as the original `Build with AI.ipynb`.

Once this works here, `generate_replies_langgraph.py` in this same folder is this exact graph, wrapped in a loop that runs it over a whole `reviews_export.json` file.

In [ ]:
!pip install anthropic langgraph -q

## Setup

One Anthropic client, reused by every agent below. In Colab this will prompt you for a key; locally, set the `ANTHROPIC_API_KEY` environment variable first and it'll skip the prompt.

In [ ]:
import json
import os
from getpass import getpass
from typing import TypedDict

import anthropic
from langgraph.graph import StateGraph, END

api_key = os.environ.get("ANTHROPIC_API_KEY") or getpass("Anthropic API key: ")
client = anthropic.Anthropic(api_key=api_key)
MODEL = "claude-haiku-4-5-20251001"


## Two small helpers every agent will use

- `ask_claude` sends one prompt, returns the text back.
- `parse_json` pulls the `{...}` out of that text — every agent below asks Claude to reply in strict JSON, so this is how we turn that text into a real Python dict.

In [ ]:
def ask_claude(prompt: str) -> str:
    message = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text.strip()


def parse_json(text: str) -> dict:
    start = text.find("{")
    end = text.rfind("}") + 1
    try:
        return json.loads(text[start:end])
    except Exception:
        return {}

# sanity check
ask_claude("Say hello in exactly 3 words.")


## The state

LangGraph passes **one dict** from agent to agent — this is the "state". Each agent reads whatever fields it needs off it, and returns only the fields it wants to change; LangGraph merges that into the state before handing it to the next agent.

`TypedDict` here is just documentation — it lists every field any agent might read or write, so it's easy to see the whole shape in one place.

In [ ]:
class ReviewState(TypedDict):
    review_text: str
    needs_human: bool
    triage_reason: str
    analysis: dict
    draft_reply: str
    critic_feedback: str
    approved: bool
    revisions: int
    status: str        # "shipped" or "escalated"
    final_reply: str


## Agent 1 — Triage

Before any AI drafts a reply, check whether this review is even safe to auto-respond to. Illness, contamination, legal threats, a minor's safety — those go straight to a human, no AI draft at all.

An agent here is nothing special: it's a plain function that takes the state, asks Claude one question, and returns a couple of fields to update.

In [ ]:
def triage_node(state: ReviewState) -> dict:
    prompt = f"""You are screening a customer review for a boba cafe before an AI drafts a reply.

Flag "needs_human" as true ONLY if the review mentions any of:
- illness or an allergic reaction
- a foreign object or contamination in the drink/food
- a threat of legal action
- a minor's safety

Return ONLY JSON: {{"needs_human": true or false, "reason": "short reason, or empty string"}}

Review:
{state['review_text']}"""
    result = parse_json(ask_claude(prompt))
    return {
        "needs_human": result.get("needs_human", False),
        "triage_reason": result.get("reason", ""),
    }


# try it on two very different reviews
safe_review = "The taro milk tea was way too sweet today and service was pretty slow."
unsafe_review = "I found a piece of plastic in my drink and my daughter nearly choked on it."

print(triage_node({"review_text": safe_review}))
print(triage_node({"review_text": unsafe_review}))


## Agent 2 — Analyst

This is the original pipeline's job, unchanged: pull out overall + per-aspect sentiment (Food Quality / Service / Ambience), each with a couple of supporting phrases.

In [ ]:
def analyst_node(state: ReviewState) -> dict:
    prompt = f"""Analyze this boba cafe review. For Overall, Food Quality, Service, and
Ambience, give a sentiment ("Positive"/"Negative"/"Neutral"/"Not Applicable")
and up to 2 short supporting phrases (empty list if the aspect isn't mentioned).

Return ONLY JSON in this shape:
{{
  "Overall": "label",
  "Food Quality": {{"sentiment": "label", "features": []}},
  "Service": {{"sentiment": "label", "features": []}},
  "Ambience": {{"sentiment": "label", "features": []}}
}}

Review:
{state['review_text']}"""
    return {"analysis": parse_json(ask_claude(prompt))}


analysis_result = analyst_node({"review_text": safe_review})
analysis_result


## Agent 3 — Drafter

Writes the manager reply from the analyst's findings. If the critic (next agent) rejected an earlier attempt, its feedback gets folded into the prompt so this is a *revision*, not a fresh guess each time.

In [ ]:
def drafter_node(state: ReviewState) -> dict:
    revision_note = ""
    if state.get("critic_feedback"):
        revision_note = f"\nYour last draft was rejected for this reason - fix it: {state['critic_feedback']}"

    prompt = f"""Write a manager reply (max 3 sentences) to this boba cafe review, based on
the analysis below. Be warm, professional, and specific. Never say
"we apologize for any inconvenience."

Review:
{state['review_text']}

Analysis:
{json.dumps(state['analysis'])}
{revision_note}

Return ONLY JSON: {{"reply": "your drafted reply"}}"""
    result = parse_json(ask_claude(prompt))
    return {"draft_reply": result.get("reply", "")}


draft_result = drafter_node({"review_text": safe_review, "analysis": analysis_result["analysis"], "critic_feedback": ""})
draft_result


## Agent 4 — Critic

Before a reply ships, a second pass checks it against a small rubric — no promised refunds, no admitting legal fault, and it has to actually respond to what the customer said. This is what makes the pipeline genuinely *multi-agent* rather than a renamed single call: the critic can disagree and send work back.

In [ ]:
def critic_node(state: ReviewState) -> dict:
    prompt = f"""You are a brand/compliance reviewer for a boba cafe. Check this draft
manager reply against 3 rules:
1. It does not promise a refund, free item, or compensation.
2. It does not admit legal fault.
3. It actually responds to the specific thing the customer said (not generic).

Review: {state['review_text']}
Draft reply: {state['draft_reply']}

Return ONLY JSON: {{"approved": true or false, "feedback": "what to fix, or empty string if approved"}}"""
    result = parse_json(ask_claude(prompt))
    return {
        "approved": result.get("approved", False),
        "critic_feedback": result.get("feedback", ""),
        "revisions": state.get("revisions", 0) + 1,
    }


critic_node({"review_text": safe_review, "draft_reply": draft_result["draft_reply"], "revisions": 0})


## The two terminal nodes, and the routing decisions

`ship_node` and `escalate_node` just record the outcome — nothing fancy.

The interesting part is the two **routing functions**. In LangGraph these are plain Python functions too: they look at the state and return the *name* of whichever node should run next. That's the entire mechanism behind both decision points in the diagram at the top.

In [ ]:
def ship_node(state: ReviewState) -> dict:
    return {"status": "shipped", "final_reply": state["draft_reply"]}


def escalate_node(state: ReviewState) -> dict:
    return {"status": "escalated", "final_reply": ""}


MAX_REVISIONS = 2

def after_triage(state: ReviewState) -> str:
    return "escalate" if state["needs_human"] else "analyst"


def after_critic(state: ReviewState) -> str:
    if state["approved"]:
        return "ship"
    if state["revisions"] < MAX_REVISIONS:
        return "drafter"
    return "escalate"


## Wiring it all into a graph

This is the actual LangGraph part — everything above was just Python functions. `StateGraph` takes those functions and connects them:

- `add_node(name, fn)` registers a function as a node.
- `add_edge(a, b)` means "after a, always run b".
- `add_conditional_edges(a, router_fn, {...})` means "after a, call `router_fn` and go to whichever node it names".
- `compile()` turns the graph definition into something you can actually run.

In [ ]:
def build_graph():
    graph = StateGraph(ReviewState)

    graph.add_node("triage", triage_node)
    graph.add_node("analyst", analyst_node)
    graph.add_node("drafter", drafter_node)
    graph.add_node("critic", critic_node)
    graph.add_node("ship", ship_node)
    graph.add_node("escalate", escalate_node)

    graph.set_entry_point("triage")
    graph.add_conditional_edges("triage", after_triage, {"analyst": "analyst", "escalate": "escalate"})
    graph.add_edge("analyst", "drafter")
    graph.add_edge("drafter", "critic")
    graph.add_conditional_edges("critic", after_critic, {"ship": "ship", "drafter": "drafter", "escalate": "escalate"})
    graph.add_edge("ship", END)
    graph.add_edge("escalate", END)

    return graph.compile()


app = build_graph()


## Running it end to end

One call — `app.invoke(...)` — now runs all four agents (or fewer, if triage escalates) in the right order, looping the drafter and critic if needed.

In [ ]:
def new_state(review_text: str) -> ReviewState:
    return {
        "review_text": review_text, "needs_human": False, "triage_reason": "",
        "analysis": {}, "draft_reply": "", "critic_feedback": "", "approved": False,
        "revisions": 0, "status": "", "final_reply": "",
    }

result = app.invoke(new_state(safe_review))
print("status:", result["status"])
print("reply: ", result["final_reply"])


In [ ]:
# the unsafe review should skip analyst/drafter/critic entirely and escalate straight away
result_unsafe = app.invoke(new_state(unsafe_review))
print("status:", result_unsafe["status"])
print("reason:", result_unsafe["triage_reason"])


## From notebook to script

Everything above — the state, the 4 agents, the 2 routing functions, `build_graph()` — is copy-pasted as-is into `generate_replies_langgraph.py` in this folder. The only thing the script adds is a loop: read `reviews_export.json`, call `app.invoke(...)` once per review, write `manager_responses.json` in the shape the bobabean Staff Tools panel expects.

```bash
cd apps/bobabean-ai-replies/langgraph_version
pip install -r ../requirements.txt langgraph
python generate_replies_langgraph.py --input ../reviews_export.json --output manager_responses.json
```